# gitmatplotlib Demo

A walkthrough of every feature in `gitmatplotlib`. This library stamps your matplotlib plots with the current git commit hash so you can always trace which code version produced a figure.

### Features covered

1. **`stamp()`** — manually stamp a figure with git info
2. **`configure()` / `reset_config()`** — set notebook-wide defaults
3. **`enable()` / `disable()`** — auto-stamp every plot
4. **`auto_stamp()`** — context manager for scoped auto-stamping
5. **`auto_commit`** — commit changes before stamping
6. **`get_git_info()`** — inspect git state programmatically
7. **Customisation** — format strings, position, style, dirty markers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gitmatplotlib

x = np.linspace(0, 2 * np.pi, 100)

---
## 1. `stamp()` — basic usage

Call `stamp()` after creating your plot. It adds a small commit hash annotation to the current figure.

- Pass a `Figure`, `Axes`, or nothing (uses `plt.gcf()`).
- Returns the `matplotlib.text.Text` artist so you can further customise it.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, np.sin(x), label="sin(x)")
plt.plot(x, np.cos(x), label="cos(x)")
plt.legend()
plt.title("Basic stamp — commit hash in bottom-right")

gitmatplotlib.stamp()
plt.show()

### Stamping a specific figure or axes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.plot(x, np.sin(x + i))
    ax.set_title(f"Phase = {i}")

# Pass the figure explicitly — works with any number of subplots
gitmatplotlib.stamp(fig)
plt.tight_layout()
plt.show()

In [ ]:
# You can also pass an Axes — it stamps the parent figure
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(["A", "B", "C"], [3, 7, 5])
gitmatplotlib.stamp(ax)
plt.show()

---
## 2. `configure()` / `reset_config()` — notebook-wide defaults

Call `configure()` once at the top of your notebook. All subsequent `stamp()` calls (including via `enable()`) use these defaults. Any argument passed directly to `stamp()` still takes precedence.

In [ ]:
gitmatplotlib.configure(
    fmt="{repo} {branch}@{commit}{dirty}",
    fontsize=8,
    color="slategray",
    alpha=0.7,
)

In [ ]:
# stamp() now uses the configured defaults — no args needed
plt.figure(figsize=(8, 4))
plt.plot(x, np.sin(x))
plt.title("Uses configured defaults")

gitmatplotlib.stamp()
plt.show()

In [ ]:
# Override just one option — the rest still come from configure()
plt.figure(figsize=(8, 4))
plt.plot(x, np.cos(x))
plt.title("Override just the color to red")

gitmatplotlib.stamp(color="red")
plt.show()

In [ ]:
# Reset back to built-in defaults
gitmatplotlib.reset_config()

plt.figure(figsize=(8, 4))
plt.plot(x, np.sin(x) ** 2)
plt.title("After reset_config() — back to built-in defaults")

gitmatplotlib.stamp()
plt.show()

---
## 3. `enable()` / `disable()` — auto-stamp every plot

Call `enable()` once and all subsequent figures are stamped automatically — no explicit `stamp()` call needed.

Works with `plt.show()`, `fig.savefig()`, and Jupyter's inline backend. Also picks up `configure()` defaults.

In [ ]:
gitmatplotlib.enable(fmt="{commit}{dirty}", color="darkgreen", fontsize=9)

In [ ]:
# No stamp() call — it's automatic
plt.figure(figsize=(8, 4))
plt.scatter(np.random.randn(50), np.random.randn(50))
plt.title("Auto-stamped via enable()")
plt.show()

In [ ]:
# Also works with savefig — stamp is applied before saving
plt.figure(figsize=(8, 4))
plt.plot(x, np.exp(-x / 3))
plt.title("Auto-stamped on savefig")
plt.savefig("/tmp/gitmatplotlib_savefig_test.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
gitmatplotlib.disable()

In [ ]:
# After disable(), plots are no longer auto-stamped
plt.figure(figsize=(8, 4))
plt.plot(x, x ** 2)
plt.title("After disable() — no stamp")
plt.show()

---
## 4. `auto_stamp()` — context manager

Auto-stamp figures within a `with` block only. Useful when you want auto-stamping for a specific section rather than the whole notebook.

In [ ]:
with gitmatplotlib.auto_stamp(fontsize=9, color="purple"):
    plt.figure(figsize=(8, 4))
    plt.plot(x, np.sin(x) * np.cos(x))
    plt.title("Inside auto_stamp() context — stamped")
    plt.show()

# Outside the block — no stamp
plt.figure(figsize=(8, 4))
plt.plot(x, np.sin(x) + np.cos(x))
plt.title("Outside auto_stamp() context — no stamp")
plt.show()

---
## 5. `auto_commit` — commit before stamping

Use `auto_commit=True` to automatically `git add -A && git commit` before reading the commit hash. This ensures the stamp always points to a clean snapshot of the code that produced the plot.

If the tree is already clean, no commit is created.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, np.sin(x) * np.exp(-x / 10))
plt.title("auto_commit=True — always a clean hash")

gitmatplotlib.stamp(auto_commit=True)
plt.show()

In [ ]:
# Set auto_commit as a default for all stamps
gitmatplotlib.configure(auto_commit=True)

plt.figure(figsize=(8, 4))
plt.plot(x, np.cos(x) * np.exp(-x / 10))
plt.title("auto_commit via configure()")

gitmatplotlib.stamp()
plt.show()

gitmatplotlib.reset_config()

In [ ]:
# Or combine with enable() for fully automatic commit + stamp
# gitmatplotlib.enable(auto_commit=True)

In [ ]:
# auto_commit() is also available as a standalone function
created = gitmatplotlib.auto_commit()
print(f"Commit created: {created}")

---
## 6. `get_git_info()` — inspect git state

Returns a `GitInfo` dataclass with:
- `commit` — short hash (e.g. `a1b2c3d`)
- `dirty` — `True` if there are uncommitted changes
- `branch` — current branch name (`None` if detached HEAD)
- `repo` — `owner/repo` from remote URL, or directory name

In [ ]:
info = gitmatplotlib.get_git_info()

print(f"Commit:  {info.commit}")
print(f"Dirty:   {info.dirty}")
print(f"Branch:  {info.branch}")
print(f"Repo:    {info.repo}")

In [ ]:
# Format into a custom label string
print(info.label())  # default: "{commit}{dirty}"
print(info.label(fmt="{repo} {branch}@{commit}{dirty}"))
print(info.label(fmt="{commit}{dirty}", dirty_marker="!"))  # custom dirty marker

In [ ]:
# You can pass pre-fetched git_info to stamp() to avoid repeated subprocess calls
info = gitmatplotlib.get_git_info()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(x, np.sin(x))
ax1.set_title("Plot A")
ax2.plot(x, np.cos(x))
ax2.set_title("Plot B")

gitmatplotlib.stamp(fig, git_info=info)
plt.tight_layout()
plt.show()

---
## 7. Customisation — format, position, style

All these options work on both `stamp()` and `configure()`.

### Format string

Placeholders: `{commit}`, `{dirty}`, `{branch}`, `{repo}`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

formats = [
    "{commit}{dirty}",
    "{branch}@{commit}{dirty}",
    "{repo} {branch}@{commit}{dirty}",
]
for ax, fmt in zip(axes, formats):
    ax.plot(x, np.sin(x))
    ax.set_title(f'fmt="{fmt}"', fontsize=9)

# Stamp each subplot's figure with a different format
# (they share one figure, so we stamp once with the last format)
gitmatplotlib.stamp(fig, fmt="{repo} {branch}@{commit}{dirty}", fontsize=9)
plt.tight_layout()
plt.show()

### Position presets

In [ ]:
positions = {
    "bottom-right (default)": dict(pos=(0.99, 0.01), ha="right", va="bottom"),
    "bottom-left":            dict(pos=(0.01, 0.01), ha="left",  va="bottom"),
    "top-right":              dict(pos=(0.99, 0.99), ha="right", va="top"),
    "top-left":               dict(pos=(0.01, 0.99), ha="left",  va="top"),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, kwargs) in zip(axes.flat, positions.items()):
    ax.plot(x, np.sin(x))
    ax.set_title(name)

# Demonstrate all four corner positions on separate figures
for name, kwargs in positions.items():
    fig_pos, ax_pos = plt.subplots(figsize=(6, 3))
    ax_pos.plot(x, np.sin(x))
    ax_pos.set_title(name)
    gitmatplotlib.stamp(fig_pos, **kwargs)
    plt.tight_layout()
    plt.show()

### Style options

In [ ]:
styles = [
    dict(fontsize=6,  color="gray",  alpha=0.3, label="Subtle"),
    dict(fontsize=10, color="gray",  alpha=0.5, label="Default"),
    dict(fontsize=14, color="blue",  alpha=0.8, label="Bold"),
    dict(fontsize=10, color="red",   alpha=1.0, label="Loud"),
]

for style in styles:
    label = style.pop("label")
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(x, np.sin(x))
    ax.set_title(f"Style: {label}")
    gitmatplotlib.stamp(fig, **style)
    plt.tight_layout()
    plt.show()

### Custom dirty marker

In [ ]:
# The default dirty marker is " (dirty)"
# You can change it to anything:

plt.figure(figsize=(8, 4))
plt.plot(x, np.sin(x))
plt.title('dirty_marker="*"')
gitmatplotlib.stamp(dirty_marker="*")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(x, np.cos(x))
plt.title('dirty_marker=" [UNCOMMITTED]"')
gitmatplotlib.stamp(dirty_marker=" [UNCOMMITTED]")
plt.show()

### `strict` mode

In [ ]:
# By default, stamp() silently does nothing if not in a git repo.
# Use strict=True to raise a GitNotFoundError instead:

try:
    gitmatplotlib.stamp(repo_path="/tmp", strict=True)
except gitmatplotlib.GitNotFoundError as e:
    print(f"Caught error: {e}")

---
## Summary

| Function | Description |
|---|---|
| `stamp(target, ...)` | Add git info annotation to a figure |
| `configure(...)` | Set default options for all subsequent stamps |
| `reset_config()` | Reset defaults back to built-in values |
| `enable(...)` | Auto-stamp all plots (hooks `show`, `savefig`, and IPython) |
| `disable()` | Stop auto-stamping |
| `auto_stamp(...)` | Context manager for scoped auto-stamping |
| `auto_commit(...)` | Stage and commit all changes if the tree is dirty |
| `get_git_info(...)` | Get a `GitInfo` object with commit, dirty, branch, repo |